In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# ==========================================
# 1. DATA: INGESTION & PIPELINE ASSEMBLY
# ==========================================
print("--- STAGE 1: DATA INGESTION & CLEANING ---")

student_info = pd.read_csv("studentInfo.csv")
student_assess = pd.read_csv("studentAssessment.csv")
assessments = pd.read_csv("assessments.csv")
student_vle = pd.read_csv("studentVle.csv", engine='python', on_bad_lines='warn')

# Define target variable: Drop-out/Failure (1) vs. Success (0)
student_info['target_at_risk'] = student_info['final_result'].apply(
    lambda x: 1 if x in ['Withdrawn', 'Fail'] else 0
)

# ==========================================
# 2. FEATURE ENGINEERING (PREDICTIVE INDICATORS)
# ==========================================
print("\n--- STAGE 2: FEATURE ENGINEERING ---")

# Feature 1: Average Assessment Score
assess_merged = pd.merge(student_assess, assessments, on='id_assessment')
student_scores = assess_merged.groupby('id_student')['score'].mean().reset_index()
student_scores.rename(columns={'score': 'avg_assessment_score'}, inplace=True)

# Feature 2: Total VLE Engagement (Clicks)
vle_clicks = student_vle.groupby('id_student')['sum_click'].sum().reset_index()
vle_clicks.rename(columns={'sum_click': 'total_lms_clicks'}, inplace=True)

# Merge features into master dataset
model_df = pd.merge(student_info, student_scores, on='id_student', how='left')
model_df = pd.merge(model_df, vle_clicks, on='id_student', how='left')

# Handle missing values
model_df['avg_assessment_score'] = model_df['avg_assessment_score'].fillna(model_df['avg_assessment_score'].median())
model_df['total_lms_clicks'] = model_df['total_lms_clicks'].fillna(0)

# Encode demographic features
model_df['disability_encoded'] = model_df['disability'].apply(lambda x: 1 if x == 'Y' else 0)

# Select features for machine learning
feature_cols = ['num_of_prev_attempts', 'studied_credits', 'avg_assessment_score', 'total_lms_clicks', 'disability_encoded']
X = model_df[feature_cols]
y = model_df['target_at_risk']

print(f"Dataset ready! Total records: {X.shape[0]} | Features count: {len(feature_cols)}")

# ==========================================
# 3. INSIGHT: MODEL TRAINING & EVALUATION
# ==========================================
print("\n--- STAGE 3: MODEL TRAINING & EVALUATION ---")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# Model Performance Evaluation
y_pred = model.predict(X_test)
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"\n Model ROC-AUC Score: {roc_auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Feature Importance Breakdown
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n--- LEADING INDICATORS OF STUDENT RISK ---")
print(importances.to_string(index=False))

# ==========================================
# 4. IMPACT: AUTOMATED RISK TIERING REPORT
# ==========================================
print("\n--- STAGE 4: GENERATING AUTOMATED RISK REPORT ---")

model_df['risk_probability'] = model.predict_proba(X)[:, 1]

def assign_tier(prob):
    if prob >= 0.70:
        return 'HIGH RISK (Immediate Intervention)'
    elif prob >= 0.40:
        return 'MEDIUM RISK (Monitor Closely)'
    else:
        return 'LOW RISK (On Track)'

model_df['risk_tier'] = model_df['risk_probability'].apply(assign_tier)

# Export high-risk student list
high_risk_students = model_df[model_df['risk_tier'] == 'HIGH RISK (Immediate Intervention)'][
    ['id_student', 'code_module', 'code_presentation', 'risk_probability', 'avg_assessment_score', 'total_lms_clicks']
]

high_risk_students.to_csv("high_risk_student_alerts.csv", index=False)

print(f"\n Automated alert report generated! {len(high_risk_students)} high-risk students flagged.")
print(" Report saved to Colab files: 'high_risk_student_alerts.csv'")

--- STAGE 1: DATA INGESTION & CLEANING ---

--- STAGE 2: FEATURE ENGINEERING ---
Dataset ready! Total records: 32593 | Features count: 5

--- STAGE 3: MODEL TRAINING & EVALUATION ---

 Model ROC-AUC Score: 0.8714

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.80      0.77      3077
           1       0.81      0.77      0.79      3442

    accuracy                           0.78      6519
   macro avg       0.78      0.78      0.78      6519
weighted avg       0.78      0.78      0.78      6519


--- LEADING INDICATORS OF STUDENT RISK ---
             Feature  Importance
avg_assessment_score    0.624083
    total_lms_clicks    0.323620
     studied_credits    0.032582
num_of_prev_attempts    0.016882
  disability_encoded    0.002833

--- STAGE 4: GENERATING AUTOMATED RISK REPORT ---

 Automated alert report generated! 10433 high-risk students flagged.
 Report saved to Colab files: 'high_risk_student_alerts.csv'
